In [1]:
import pandas as pd
import sys
import os
from IPython.display import display
from tqdm.auto import tqdm
tqdm.pandas()

sys.path.append(os.path.abspath("../../"))
from optimizer import variables

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [2]:
def heuristic_price_prediction(price_df,operating_protocol_transform, operating_protocol_window_size, apply_blend):

    

    def _apply_transform(operating_protocol_transform):
        
        
        # Mode 1) Actual prices
        # shift(1) so each seasonal group's signal is derived from the previous seasonal group's mean.
        # The NaN row produced by shift(1) falls in the extra leading 14-day buffer and is discarded by _filter().
        if operating_protocol_transform == "None":
            df = pd.pivot_table(price_df,values='Price',index=["Year", "Seasonal_group"],columns="Interval",aggfunc="mean").sort_index()
            df = df.shift(1)

        # Mode 2) Smoothed prices (exponentially weighted moving average)
        # ewm().mean() for seasonal group t uses groups 0..t; shift(1) makes group t use groups 0..t-1 only — causal.
        elif operating_protocol_transform == "Smoothed":
            df = pd.pivot_table(price_df,values="Price",index=["Year", "Seasonal_group"],columns="Interval",aggfunc="mean").sort_index()
            df = df.ewm(alpha=0.1).mean().shift(1)

        # Mode 3) Pessimistic prices (5th percentile)
        # shift(1) so each seasonal group's signal is derived from the previous seasonal group's quantile.
        elif operating_protocol_transform == "Pessimistic":
            df = pd.pivot_table(price_df,values='Price',index=["Year", "Seasonal_group"],columns="Interval",aggfunc=lambda x: x.quantile(0.05)).sort_index()
            df = df.shift(1)

        # Mode 4) Pessimistic smoothed prices
        elif operating_protocol_transform == "Pessimistic_Smoothed":
            df = pd.pivot_table(price_df,values='Price',index=["Year", "Seasonal_group"],columns="Interval",aggfunc=lambda x: x.quantile(0.05)).sort_index()
            df = df.ewm(alpha=0.1).mean().shift(1)

        return df

    def _apply_blend(operating_protocol_window_size):
        # Use a separate original copy so that updating interval i does not cascade into interval i+1.
        # Use a backward-looking window [i, i-1, i-2, ...] so interval i is never smoothed using
        # a future interval's price.
        original = transformed_df.copy()
        df = transformed_df.copy()
        for i, protocol_interval in enumerate(df.columns):
            sliding_window_of_x_protocol_intervals = [
                original.columns[
                    (i - offset) % len(original.columns)
                    ] for offset in range(int(operating_protocol_window_size))
                ]
            df[protocol_interval] = original[sliding_window_of_x_protocol_intervals].mean(axis=1) 
        
        return df
        
     
    
    def _create_predicted_price():
        df = (transformed_df.stack().rename("Predicted_price").reset_index())
        df = price_df.copy().merge(df.copy(), on=["Year", "Seasonal_group", "Interval"], how="left")
        df = df[['Date','Predicted_price']]
        return df
    

    def _create_direct_bess_instructions():
       
        def _translate_price_to_rank():
            # No additional shift(1) here — all transforms now apply shift(1) themselves,
            # so transformed_df for seasonal group t already reflects the previous seasonal group's data.
            df = (transformed_df.rank(axis=1, method="first").dropna(how="any").astype(int))
            df = (df.stack().rename("Rank").reset_index())
            df.columns = ["Year", "Seasonal_group", "Interval", "Rank"]

            return df

        def _translate_rank_to_bess_instruction(rank_df):
            number_of_intervals_to_instruct = int(variables.bess_hours * 60 / variables.operation_granularity_in_minutes)
            n_intervals = len(transformed_df.columns)
            df = price_df.copy().merge(rank_df, on=["Year", "Seasonal_group", "Interval"], how="left")
            df["BESS_instruction"] = 0
            df.loc[df["Rank"] <= number_of_intervals_to_instruct, "BESS_instruction"] = 1 # Discharge 
            df.loc[df["Rank"] > n_intervals - number_of_intervals_to_instruct, "BESS_instruction"] = 2 # Charge
            df = df[['Date','BESS_instruction']]

            return df
        
        rank_df = _translate_price_to_rank()
        df = _translate_rank_to_bess_instruction(rank_df)

        return df

    def _filter():
        operation_start_date = pd.to_datetime(variables.operation_start_date, format="%d/%m/%Y %H:%M")
        operation_end_date = pd.to_datetime(variables.operation_end_date, format="%d/%m/%Y %H:%M")

        df = Predicted_price.merge(BESS_instruction, on="Date", how="left")
        df = df.loc[(df["Date"] >= operation_start_date)& (df["Date"] <= operation_end_date)].copy()
        return df.reset_index(drop=True)

    
    if apply_blend:
        transformed_df = _apply_transform(operating_protocol_transform)
        transformed_df = _apply_blend(operating_protocol_window_size)
    else:
        transformed_df = _apply_transform(operating_protocol_transform)
    

    Predicted_price = _create_predicted_price()
    BESS_instruction = _create_direct_bess_instructions()
    heuristic_predicted_price_df = _filter()

    return heuristic_predicted_price_df


Comparing multiple

In [3]:
if False:
        operating_protocol_seasonal_grouping_granularity = ["4D"]
        operating_protocol_transform = ["Smoothed"]
        operating_protocol_window_size = [2]
        apply_blend = [True,False]

        items = [
                (a, b, c, d)
                for a in operating_protocol_seasonal_grouping_granularity
                for b in operating_protocol_transform
                for c in operating_protocol_window_size
                for d in apply_blend
        ]

        actual_price = pd.read_csv("../1_Dataset/2_Processed_data/price.csv")
        actual_price["Date"] = pd.to_datetime(actual_price["Date"])

        comp_dict = {"actual_price": actual_price}

        for a, b, c, d in items:
                p = heuristic_price_prediction(
                        price_df_path = "../1_Dataset/2_Processed_data/price_extra_14_days.csv",
                        operating_protocol_seasonal_grouping_granularity = a,
                        operating_protocol_transform = b,
                        operating_protocol_window_size = c,
                        apply_blend = d
                )
        key = f"{a}|{b}|w{c}|blend={d}"
        comp_dict[key] = p




        from IPython.display import HTML

        df_combined = pd.concat([df.set_index("Date").iloc[:, 0].rename(name) for name, df in comp_dict.items()], axis=1)
        styled_df = df_combined.head(100).style.background_gradient(cmap="RdYlGn", axis=None).format("{:.2f}")
        styled_df.to_excel("Matrix.xlsx")
        HTML(styled_df.to_html())


In [4]:
def _data_prep(price_df_path, operating_protocol_seasonal_grouping_granularity):
        df = pd.read_csv(price_df_path)
        df["Date"] = pd.to_datetime(df["Date"])
        df["Year"] = df["Date"].dt.year
        df["Seasonal_group"] = (df.groupby(pd.Grouper(key="Date", freq=operating_protocol_seasonal_grouping_granularity)).ngroup() + 1)
        df["Interval"] = (df["Date"].dt.hour * (60 // int(variables.operation_granularity_in_minutes)) + df["Date"].dt.minute // int(variables.operation_granularity_in_minutes)+ 1)
        
        return df


if variables.run_multiple_optimisations:
        price_df_wide = _data_prep("../1_Dataset/2_Processed_data/resampled_aurora_price.csv", variables.operating_protocol_seasonal_grouping_granularity)

        scenario_cols = [
                col for col in price_df_wide.columns
                if col not in ["Date", "Year", "Seasonal_group", "Interval"]
        ]

        predicted_price_series = []

        # Run the heuristic once per sampled price column.
        for scenario_col in tqdm(scenario_cols):
                scenario_df = price_df_wide[["Date", "Year", "Seasonal_group", "Interval", scenario_col]].rename(
                        columns={scenario_col: "Price"}
                )

                predicted_price = heuristic_price_prediction(
                        price_df = scenario_df,
                        operating_protocol_transform = variables.operating_protocol_transform,
                        operating_protocol_window_size = variables.operating_protocol_window_size,
                        apply_blend = variables.apply_blend
                )

                # Keep output in the same wide scenario format as the input.
                predicted_price_series.append(
                        predicted_price.set_index("Date")["Predicted_price"].rename(scenario_col)
                )

        predicted_prices = pd.concat(predicted_price_series, axis=1).reset_index()

        # Export
        predicted_prices.to_csv("2_Processed_data/predicted_prices.csv", index=False)
        
else:
        # Call the heuristic creation function & create the predictions (from 1 price curve)
        price_df = _data_prep("../1_Dataset/2_Processed_data/price_extra_14_days.csv", variables.operating_protocol_seasonal_grouping_granularity)

         # Call function
        predicted_price = heuristic_price_prediction(
                price_df = price_df,
                operating_protocol_transform = variables.operating_protocol_transform,
                operating_protocol_window_size = variables.operating_protocol_window_size,
                apply_blend = variables.apply_blend
        )
        # Export
        predicted_price.to_csv("2_Processed_data/predicted_price.csv",index=False)



  0%|          | 0/1000 [00:00<?, ?it/s]